# Notebook 2: Zero-Shot Baseline & QLoRA SFT Training (Kaggle)
**Project**: Autonomous Text-to-SQL Generator Aligned via RLVR + GRPO  
**Hardware**: Kaggle GPU (T4 x 1, P100, or 2xT4 — pinned to single GPU to avoid DataParallel + 4-bit conflicts)  
**Phase**: Phase 2 (Sandbox Verifier) & Phase 3 (Zero-Shot Baseline & Supervised Fine-Tuning)

---

In [ ]:
# Step 1: Install a pinned, compatible training stack for Kaggle GPU training
# - TRL 0.25.0 uses the standard causal-LM loss path (avoiding fused LM-head issues).
# - bitsandbytes < 0.50.0 avoids the 0.50+ gemm_4bit / cuBLASLt fallback regression on T4 (CC 7.5).
# - Pin Transformers 4.x and PEFT >= 0.18 for stable QLoRA compatibility.
!pip uninstall -y -q torchvision
!pip install -q \
    "transformers==4.57.3" \
    "trl==0.25.0" \
    "peft==0.18.1" \
    "datasets>=3.0.0,<5.0.0" \
    "accelerate>=1.2.1,<2.0.0" \
    "bitsandbytes>=0.45.2,<0.50.0" \
    "sqlglot>=26.0.0"
print("Pinned ML training packages installed. Restart the Kaggle session before running the remaining cells.")

In [ ]:
# Step 2: GPU Inspection & Path Setup (With Auto-Unpack for Kaggle Inputs)
import os
import sys

# CRITICAL: Must be set BEFORE importing torch!
# 1. Pin to GPU 0 so SFTTrainer never wraps 4-bit QLoRA in legacy nn.DataParallel on Kaggle T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 2. Reduce CUDA allocator fragmentation on 15 GB T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Fully block torchvision to prevent Kaggle CUDA/PyTorch operator ABI mismatch
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import shutil
import zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA GPU is required! Please enable GPU in Kaggle Notebook Settings."
major, minor = torch.cuda.get_device_capability(0)
print(f"Visible CUDA Devices: {torch.cuda.device_count()} (pinned to single GPU for QLoRA stability)")
print(f"CUDA Device: {torch.cuda.get_device_name(0)} (Compute Capability {major}.{minor})")
print(f"Allocated Memory: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
print(f"Hardware BF16 Supported (requires CC >= 8.0): {major >= 8}")

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUTS_DIR = WORKING_DIR / "outputs" / "sft"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Auto-unpack data from uploaded dataset (sql_grpo_data.zip, sql_grpo_processed_data.zip, etc.)
train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"

def setup_dataset():
    if train_manifest.exists() and dev_manifest.exists():
        return

    # 1. Search for any zip archive containing train_manifest.jsonl or matching data naming
    all_zips = list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))
    for z in all_zips:
        if any(k in z.name.lower() for k in ["sql_grpo", "data", "processed", "spider"]):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    names = zf.namelist()
                    if any("train_manifest.jsonl" in n for n in names):
                        print(f"📦 Unpacking dataset from archive: {z} ...")
                        has_data_root = any(n.startswith("data/") for n in names)
                        extract_target = WORKING_DIR if has_data_root else DATA_DIR
                        zf.extractall(extract_target)
                        print(f"Extracted into: {extract_target}")
                        break
            except Exception as e:
                print(f"Warning inspecting {z}: {e}")

    # 2. If archive was extracted to /kaggle/working/processed/ instead of /kaggle/working/data/processed/
    if not train_manifest.exists() and (WORKING_DIR / "processed" / "train_manifest.jsonl").exists():
        print(f"Relocating data folders to {DATA_DIR} ...")
        for folder in ["processed", "raw"]:
            src = WORKING_DIR / folder
            dst = DATA_DIR / folder
            if src.exists() and not dst.exists():
                shutil.move(src, dst)

    # 3. If Kaggle pre-extracted the dataset in /kaggle/input/ (without a zip file)
    if not train_manifest.exists():
        found_manifests = list(Path("/kaggle/input").glob("**/train_manifest.jsonl"))
        if found_manifests:
            m_path = found_manifests[0]
            dataset_root = m_path.parent.parent if m_path.parent.name == "processed" else m_path.parent
            print(f"📂 Found pre-extracted dataset at {dataset_root}. Syncing to {DATA_DIR} ...")
            src_proc = dataset_root / "processed" if (dataset_root / "processed").exists() else m_path.parent
            dst_proc = DATA_DIR / "processed"
            if src_proc.exists() and not dst_proc.exists():
                shutil.copytree(src_proc, dst_proc)
            src_raw = dataset_root / "raw" if (dataset_root / "raw").exists() else (dataset_root / "data" / "raw")
            dst_raw = DATA_DIR / "raw"
            if src_raw.exists() and not dst_raw.exists():
                shutil.copytree(src_raw, dst_raw)

setup_dataset()

if not (train_manifest.exists() and dev_manifest.exists()):
    available_inputs = [str(p) for p in Path("/kaggle/input").glob("*")]
    raise FileNotFoundError(
        "❌ CRITICAL ERROR: train_manifest.jsonl was not found in /kaggle/input or /kaggle/working!\n\n" +
        f"Items currently in /kaggle/input: {available_inputs}\n\n" +
        "👉 HOW TO FIX IN KAGGLE:\n" +
        "1. In the right-hand panel, under 'Input', click '+ Add Input'.\n" +
        "2. Select your uploaded 'sql_grpo_data' dataset (or upload 'sql_grpo_data.zip').\n" +
        "3. Re-run this cell."
    )

print(f"✅ Verified processed dataset at: {DATA_DIR}")
with open(train_manifest, "r", encoding="utf-8") as f:
    num_train = sum(1 for _ in f)
with open(dev_manifest, "r", encoding="utf-8") as f:
    num_dev = sum(1 for _ in f)
print(f"  - Train examples: {num_train}")
print(f"  - Dev examples:   {num_dev}")

In [ ]:
# Hardened Shared SQL Parser, AST Policy, Authorizer, Canonical Comparator & Sandbox
import hashlib
import json as _json
import math
import sqlite3
import time
from collections import Counter
from decimal import Decimal
from typing import Any, Dict, List, Optional, Tuple
import sqlglot
from sqlglot import exp

SAFE_FUNCTIONS = {
    "count", "sum", "avg", "min", "max", "total", "group_concat", "string_agg",
    "lower", "upper", "length", "octet_length", "substr", "substring",
    "trim", "ltrim", "rtrim", "replace", "instr", "hex", "unicode", "char", "quote",
    "round", "abs", "ceil", "ceiling", "floor", "mod", "power", "sqrt", "sign",
    "coalesce", "nullif", "ifnull", "iif",
    "strftime", "date", "time", "datetime", "julianday", "unixepoch",
    "cast", "like", "glob", "typeof",
    "rank", "dense_rank", "row_number", "ntile", "lag", "lead", "first_value", "last_value",
}
SYSTEM_TABLES = {
    "sqlite_master", "sqlite_schema", "sqlite_temp_master",
    "sqlite_temp_schema", "sqlite_sequence", "sqlite_stat1", "sqlite_stat4",
}

def extract_completion_text(c: Any) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        for msg in reversed(c):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
            if isinstance(msg, dict) and "content" in msg:
                return msg.get("content", "")
        if len(c) > 0 and isinstance(c[-1], str):
            return c[-1]
        return str(c)
    if isinstance(c, dict):
        return c.get("content", str(c))
    return str(c)

def parse_sql(text: str) -> Tuple[Optional[str], bool]:
    if not text or "<sql>" not in text or "</sql>" not in text:
        return None, False
    if text.count("<sql>") != 1 or text.count("</sql>") != 1:
        return None, False
    idx1 = text.find("<sql>") + len("<sql>")
    idx2 = text.find("</sql>")
    if idx1 >= idx2:
        return None, False
    if text[idx2 + len("</sql>"):].strip():
        return None, False
    sql = text[idx1:idx2].strip()
    if sql.startswith("```"):
        lines = sql.splitlines()[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        sql = chr(10).join(lines).strip()
    return (sql, True) if sql else (None, False)

def is_ast_safe(sql: str, valid_tables: Optional[List[str]] = None) -> bool:
    if valid_tables is not None and len(valid_tables) == 0:
        raise ValueError("is_ast_safe: valid_tables cannot be an empty list.")
    try:
        stmts = sqlglot.parse(sql, read="sqlite")
        if not stmts or len(stmts) != 1 or stmts[0] is None:
            return False
        ast = stmts[0]
        if not isinstance(ast, (exp.Select, exp.Union, exp.Intersect, exp.Except)):
            return False
        for with_node in ast.find_all(exp.With):
            if with_node.args.get("recursive"):
                return False
        for sel in ast.find_all(exp.Select):
            if not sel.expressions:
                return False
        prohibited = tuple(
            t for t in (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop,
                        getattr(exp, "AlterTable", getattr(exp, "Alter", None)),
                        exp.Pragma, exp.Command, exp.Transaction,
                        getattr(exp, "Attach", None), getattr(exp, "Detach", None),
                        getattr(exp, "Analyze", None)) if t is not None
        )
        if any(ast.find_all(prohibited)):
            return False
        for node in ast.walk():
            if isinstance(node, exp.Anonymous):
                if (node.name or "").lower() not in SAFE_FUNCTIONS:
                    return False
        cte_names = {c.alias.lower() for c in ast.find_all(exp.CTE) if c.alias}
        phys_tables = set()
        for t in ast.find_all(exp.Table):
            if isinstance(t.this, (exp.Anonymous, exp.Func)):
                return False
            if (t.catalog or "") or (t.db or "").lower() not in {"", "main"}:
                return False
            tn = t.name.lower() if t.name else ""
            if not tn or tn in cte_names:
                continue
            if tn in SYSTEM_TABLES or tn.startswith("sqlite_") or tn.startswith("pragma_"):
                return False
            phys_tables.add(tn)
        if valid_tables is not None:
            allowed = {str(t).lower() for t in valid_tables}
            if phys_tables - allowed:
                return False
        return True
    except Exception:
        return False

def create_authorizer(allowed_tables: Optional[List[str]] = None):
    norm_tbls = {t.lower() for t in allowed_tables} if allowed_tables is not None else None
    def callback(action, arg1, arg2, db, trigger):
        if action == sqlite3.SQLITE_SELECT:
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_READ:
            tname = (arg1 or "").lower()
            dname = (db or "").lower()
            if dname not in {"", "main"}:
                return sqlite3.SQLITE_DENY
            if tname in SYSTEM_TABLES or tname.startswith("sqlite_") or tname.startswith("pragma_"):
                return sqlite3.SQLITE_DENY
            if norm_tbls is not None and tname not in norm_tbls:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_FUNCTION:
            if (arg2 or "").lower() in SAFE_FUNCTIONS:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    return callback

def _canonical_val(val: Any):
    if val is None:
        return ("__NULL__", None)
    if isinstance(val, bool):
        return ("bool", val)
    if isinstance(val, int):
        return ("int", val)
    if isinstance(val, float):
        if math.isnan(val):
            return ("float_nan", "nan")
        if math.isinf(val):
            return ("float_inf", 1 if val > 0 else -1)
        return ("float", 0.0 if val == 0.0 else float(val))
    if isinstance(val, Decimal):
        return ("decimal", str(val.normalize()))
    if isinstance(val, (bytes, bytearray, memoryview)):
        return ("bytes", bytes(val).hex())
    return ("str", str(val))

def canonical_row(row: Tuple[Any, ...]):
    return tuple(_canonical_val(v) for v in row)

def compare_multiset_rows(cand_rows: List[Tuple[Any, ...]], gold_rows: List[Tuple[Any, ...]]) -> bool:
    return Counter(canonical_row(r) for r in cand_rows) == Counter(canonical_row(r) for r in gold_rows)

def execute_sandbox_query(
    db_path: "Path",
    sql: str,
    timeout_sec: float = 2.0,
    row_limit: int = 5000,
    byte_limit: int = 1_048_576,
    allowed_tables: Optional[List[str]] = None,
) -> Tuple[bool, Optional[List[Tuple]], bool, float, Optional[str]]:
    start = time.perf_counter()
    deadline = start + timeout_sec
    conn = None
    cur = None
    try:
        resolved = db_path.resolve(strict=True)
        uri = resolved.as_uri() + "?mode=ro"
        conn = sqlite3.connect(uri, uri=True, timeout=1.0)
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(create_authorizer(allowed_tables=allowed_tables))
        conn.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10_000)
        cur = conn.cursor()
        cur.execute(sql)
        rows = []
        total_bytes = 0
        truncated = False
        while True:
            rem = (row_limit + 1) - len(rows)
            if rem <= 0:
                truncated = True
                break
            batch = cur.fetchmany(min(256, rem))
            if not batch:
                break
            for r in batch:
                rows.append(r)
                total_bytes += sum(
                    len(v) if isinstance(v, (bytes, bytearray))
                    else (len(v.encode("utf-8", "replace")) if isinstance(v, str) else 8)
                    for v in r
                )
                if len(rows) > row_limit or total_bytes > byte_limit:
                    truncated = True
                    break
            if truncated:
                break
        dur_ms = (time.perf_counter() - start) * 1000.0
        if truncated:
            return False, None, True, dur_ms, "RESULT_TRUNCATED"
        return True, rows, False, dur_ms, None
    except Exception as e:
        dur_ms = (time.perf_counter() - start) * 1000.0
        return False, None, False, dur_ms, str(e)
    finally:
        if cur:
            try:
                cur.close()
            except Exception:
                pass
        if conn:
            try:
                conn.set_progress_handler(None, 0)
                conn.set_authorizer(None)
                conn.close()
            except Exception:
                pass

print("✅ Hardened Sandbox, AST Policy, and Authorizer ready.")

In [ ]:
# Step 4: Load Model & Tokenizer in 4-bit (Qwen2.5-Coder-1.5B-Instruct)
import sys
# Ensure torchvision cannot be imported by any internal transformers module
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import transformers.utils.import_utils as _imp
_imp.is_torchvision_available = lambda: False

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Detect GPU compute capability for optimal quantization (CC >= 8.0 required for native BF16 GEMM)
major, minor = torch.cuda.get_device_capability(0)
supports_bf16 = (major >= 8)
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f"CUDA Capability: {major}.{minor} | Using 4-bit compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer and 4-bit quantized model: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pin explicitly to cuda:0 (avoids multi-GPU sharding / DataParallel conflicts with 4-bit weights)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    trust_remote_code=True,
)
print(f"Model loaded successfully on cuda:0. Peak VRAM: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

In [ ]:
# Step 5: Evaluate Zero-Shot Baseline (Pass@1 Greedy) with Hardened AST + Sandbox
import json
import numpy as np

dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"
assert dev_manifest.exists(), f"dev_manifest.jsonl not found at {dev_manifest}! Please check Step 2."
with open(dev_manifest, "r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f if line.strip()][:100]
print(f"Running Zero-Shot evaluation on {len(eval_records)} dev examples...")

correct = 0
valid_sql = 0
ast_safe = 0
exec_ok_cnt = 0
lats = []
preds_log = []

for i, r in enumerate(eval_records):
    prompt_text = tokenizer.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None, pad_token_id=tokenizer.pad_token_id)
    lat_ms = (time.perf_counter() - t0) * 1000.0
    lats.append(lat_ms)
    comp = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    sql, fmt_ok = parse_sql(comp)
    is_ast = False
    ok_exec = False
    is_match = False
    if fmt_ok and sql:
        valid_sql += 1
        v_tbls = r.get("valid_tables")
        is_ast = is_ast_safe(sql, v_tbls if v_tbls else None)
        if is_ast:
            ast_safe += 1
            db_file = DATA_DIR / "raw" / "spider" / "database" / r["db_id"] / f"{r['db_id']}.sqlite"
            ok_exec, rows, trunc, _, _ = execute_sandbox_query(db_file, sql, allowed_tables=v_tbls)
            if ok_exec and not trunc and rows is not None:
                exec_ok_cnt += 1
                gold_rows = [tuple(x) if isinstance(x, list) else (x,) for x in json.loads(r["gold_result_json"])]
                if compare_multiset_rows(rows, gold_rows):
                    is_match = True
                    correct += 1
    preds_log.append({
        "example_id": r.get("example_id", f"dev_{i:06d}"),
        "db_id": r["db_id"],
        "format_valid": fmt_ok,
        "ast_safe": is_ast,
        "execution_success": ok_exec,
        "execution_match": is_match,
        "latency_ms": round(lat_ms, 2),
        "sql_extracted": sql,
    })
    if (i + 1) % 25 == 0:
        print(f"  Evaluated {i+1}/{len(eval_records)}: Current EX = {correct/(i+1)*100:.1f}%")

ex_score = correct / len(eval_records) * 100
valid_score = valid_sql / len(eval_records) * 100
ast_score = ast_safe / len(eval_records) * 100
exec_score = exec_ok_cnt / len(eval_records) * 100
print(f"🎯 Zero-Shot Baseline: EX={ex_score:.2f}%, FmtValid={valid_score:.2f}%, AST={ast_score:.2f}%, Exec={exec_score:.2f}%")

(OUTPUTS_DIR / "zero_shot_predictions.jsonl").write_text(chr(10).join(json.dumps(p) for p in preds_log) + chr(10), encoding="utf-8")
baseline_log_path = OUTPUTS_DIR / "zero_shot_baseline_metrics.json"
baseline_log_path.write_text(json.dumps({
    "num_examples": len(eval_records),
    "execution_accuracy_pct": round(ex_score, 2),
    "valid_sql_pct": round(valid_score, 2),
    "ast_safe_pct": round(ast_score, 2),
    "execution_validity_pct": round(exec_score, 2),
    "p50_latency_ms": round(float(np.percentile(lats, 50, method="linear")), 2),
    "p95_latency_ms": round(float(np.percentile(lats, 95, method="linear")), 2),
}, indent=2), encoding="utf-8")
print(f"Saved zero-shot metrics to: {baseline_log_path}")

In [ ]:
# Step 6: Configure QLoRA Adapter & SFT Dataset
import json
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# This cell must run exactly once per freshly loaded base model.
if hasattr(model, "peft_config"):
    raise RuntimeError(
        "This model already has a PEFT adapter. Restart the Kaggle session, "
        "rerun Cell 4, and execute Cell 6 only once."
    )

# Disable KV cache during training (required for gradient checkpointing)
model.config.use_cache = False

# Enable non-reentrant gradient checkpointing to save VRAM (~40% reduction) on T4
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
# Required when gradient_checkpointing=True so frozen base layers propagate grads
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

# Guarantee all trainable LoRA params are FP32 so FP16 GradScaler works safely on T4
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

model.print_trainable_parameters()
print("Trainable adapter dtypes:", sorted({str(p.dtype) for p in model.parameters() if p.requires_grad}))

train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
assert train_manifest.exists(), f"train_manifest.jsonl not found at {train_manifest}! Please check Step 2."

train_rows = []
with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        full_prompt = tokenizer.apply_chat_template(item["prompt"], tokenize=False, add_generation_prompt=True)
        # Do not manually append eos_token here: TRL 0.25.0 SFTTrainer appends EOS automatically
        full_text = full_prompt + item["completion"]
        train_rows.append({"text": full_text})

train_dataset = Dataset.from_list(train_rows)
print(f"Loaded {len(train_dataset)} training examples.")
assert len(train_dataset) >= 5000, f"Expected ~7,000 training examples, but only found {len(train_dataset)}! Aborting to prevent incomplete training."

In [ ]:
# Step 7: Train QLoRA SFT Baseline on Kaggle GPU (Single-GPU T4-safe settings)
import inspect
import transformers, trl, peft, bitsandbytes

print(
    f"Package versions | transformers={transformers.__version__} | "
    f"trl={trl.__version__} | peft={peft.__version__} | bitsandbytes={bitsandbytes.__version__}"
)

assert torch.cuda.device_count() == 1, (
    f"Detected {torch.cuda.device_count()} visible GPUs. Restart the Kaggle session and run Cell 2 "
    "before importing torch so CUDA_VISIBLE_DEVICES='0' prevents DataParallel wrapping."
)

major, minor = torch.cuda.get_device_capability(0)
# Ampere+ (CC >= 8.0) -> BF16; Turing/T4 (CC 7.5) -> FP16 AMP with FP32 LoRA weights
use_bf16 = (major >= 8)
use_fp16 = not use_bf16
print(
    f"CUDA Capability: {major}.{minor} | Training precision: "
    f"bf16={use_bf16}, fp16={use_fp16} | Optimizer: paged_adamw_8bit | Visible GPUs: {torch.cuda.device_count()}"
)

# Memory-safe training config for T4 (15 GB VRAM)
# Effective batch size = per_device(2) * grad_accum(8) = 16
# If you hit OOM on longer sequences, lower per_device_train_batch_size to 1 and raise gradient_accumulation_steps to 16.
raw_args = {
    "output_dir": str(OUTPUTS_DIR),
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "optim": "paged_adamw_8bit",
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 20,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "fp16": use_fp16,
    "bf16": use_bf16,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "dataset_text_field": "text",
    "packing": False,
    "report_to": "none",
}

# TRL 0.25.0 uses max_length; keep signature check for version resilience
sig_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sig_params:
    raw_args["max_length"] = 1024
elif "max_seq_length" in sig_params:
    raw_args["max_seq_length"] = 1024

has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig_params.values())
valid_args = raw_args if has_var_keyword else {k: v for k, v in raw_args.items() if k in sig_params}
training_args = SFTConfig(**valid_args)

# Resolve processing_class vs tokenizer across TRL versions
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "args": training_args,
}
trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Starting QLoRA SFT Training...")
trainer.train()

# Save trained SFT adapter
sft_adapter_path = OUTPUTS_DIR / "best_adapter"
trainer.model.save_pretrained(str(sft_adapter_path))
tokenizer.save_pretrained(str(sft_adapter_path))
print(f"Saved SFT Adapter to: {sft_adapter_path}")

In [ ]:
# Step 7b: Evaluate Trained QLoRA SFT Baseline on Identical Held-Out Dev Slice
SFT_EVAL_MODE = "first_100"  # Set to "full" to evaluate all 1,034 Spider dev examples
with open(DATA_DIR / "processed" / "dev_manifest.jsonl", "r", encoding="utf-8") as f:
    all_dev = [json.loads(line) for line in f if line.strip()]
sft_eval_records = all_dev if SFT_EVAL_MODE == "full" else all_dev[:100]

print(f"Running Standalone QLoRA SFT evaluation ({SFT_EVAL_MODE}: {len(sft_eval_records)} examples)...")
trainer.model.eval()
sft_correct = sft_fmt = sft_ast = sft_exec = 0
sft_lats = []
sft_preds = []

for i, r in enumerate(sft_eval_records):
    prompt_text = tokenizer.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = trainer.model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None, pad_token_id=tokenizer.pad_token_id)
    lat_ms = (time.perf_counter() - t0) * 1000.0
    sft_lats.append(lat_ms)
    comp = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    sql, fmt_ok = parse_sql(comp)
    is_ast = ok_exec = is_match = False
    if fmt_ok and sql:
        sft_fmt += 1
        v_tbls = r.get("valid_tables")
        is_ast = is_ast_safe(sql, v_tbls if v_tbls else None)
        if is_ast:
            sft_ast += 1
            db_file = DATA_DIR / "raw" / "spider" / "database" / r["db_id"] / f"{r['db_id']}.sqlite"
            ok_exec, rows, trunc, _, _ = execute_sandbox_query(db_file, sql, allowed_tables=v_tbls)
            if ok_exec and not trunc and rows is not None:
                sft_exec += 1
                gold_rows = [tuple(x) if isinstance(x, list) else (x,) for x in json.loads(r["gold_result_json"])]
                if compare_multiset_rows(rows, gold_rows):
                    is_match = True
                    sft_correct += 1
    sft_preds.append({
        "example_id": r.get("example_id", f"dev_{i:06d}"),
        "db_id": r["db_id"],
        "format_valid": fmt_ok,
        "ast_safe": is_ast,
        "execution_success": ok_exec,
        "execution_match": is_match,
        "latency_ms": round(lat_ms, 2),
        "sql_extracted": sql,
    })

sft_metrics = {
    "eval_mode": SFT_EVAL_MODE,
    "num_examples": len(sft_eval_records),
    "execution_accuracy_pct": round(sft_correct / len(sft_eval_records) * 100, 2),
    "format_validity_pct": round(sft_fmt / len(sft_eval_records) * 100, 2),
    "ast_safe_pct": round(sft_ast / len(sft_eval_records) * 100, 2),
    "execution_validity_pct": round(sft_exec / len(sft_eval_records) * 100, 2),
    "p50_latency_ms": round(float(np.percentile(sft_lats, 50, method="linear")), 2),
    "p95_latency_ms": round(float(np.percentile(sft_lats, 95, method="linear")), 2),
}
print("🎯 Standalone SFT Baseline Results:", json.dumps(sft_metrics, indent=2))
(OUTPUTS_DIR / "sft_eval_predictions.jsonl").write_text(chr(10).join(json.dumps(p) for p in sft_preds) + chr(10), encoding="utf-8")
(OUTPUTS_DIR / "sft_baseline_metrics.json").write_text(json.dumps(sft_metrics, indent=2), encoding="utf-8")

In [ ]:
# ==========================================================================
# FINAL CELL: MANDATORY ARTIFACT ZIPPER
# Zips the trained SFT adapter and evaluation logs for download
# ==========================================================================
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

output_zip_path = WORKING_DIR / "sql_grpo_sft_artifacts.zip"
print(f"Creating artifact zip at: {output_zip_path} ...")

with zipfile.ZipFile(output_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUTS_DIR):
        for file in files:
            p = Path(root) / file
            zipf.write(p, arcname=str(p.relative_to(WORKING_DIR)))

zip_size_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
print(f"✅ Successfully created: {output_zip_path.name} ({zip_size_mb:.2f} MB)")
print("\n👉 Click the link below to download directly in your browser:")
display(FileLink("sql_grpo_sft_artifacts.zip"))